# Opdracht 4 – ETL Implementatie Data Warehouse
## Robbert & Mees

In deze opdracht implementeren wij de ETL-processen van het Data Warehouse. 
We laden data vanuit het Source Data Model (SDM) naar het DWH en passen 
Slowly Changing Dimensions Type 1 en Type 2 toe.

## Inlaadstrategie

In deze opdracht maken wij gebruik van de **Incremental Data Loading** strategie.

Reden:
- we willen alleen nieuwe en gewijzigde data verwerken
- nodig voor SCD Type 2 (historie behouden)


Setup & Connecties (NOG INVULLEN)

In [1]:
import pandas as pd
import sqlite3
from loguru import logger
from datetime import datetime # nodig voor dimensies (begin/eindtijd)

sdm_conn = sqlite3.connect('../Source Data Model/BikeToDrive_V3_OLTP.db', timeout=30)
dwh_conn = sqlite3.connect('../Data Warehouse/BikeToDrive_DWH_V3.db', timeout=30)
sdm_cursor = sdm_conn.cursor()
dwh_cursor = dwh_conn.cursor()

logger.info("SDM en DWH connecties succesvol.")

2026-04-01 12:05:30.428 | INFO     | __main__:<module>:11 - SDM en DWH connecties succesvol.


Dim_Klant (SCD TYPE 2) ROBBERT

In [2]:
# ETL Dim_Klant - SCD Type 2
# Doel:
# We vullen Dim_Klant in het DWH met SCD Type 2 logica.
#
#
# SCD Type 2-regels:
# - record bestaat nog niet in DWH -> INSERT
# - record bestaat wel, maar attribuutwaarden zijn gewijzigd
#     -> oude versie afsluiten + nieuwe versie INSERT
# - record bestaat wel en is ongewijzigd -> niets doen
#
# Business key  = (klantnr, source_id)
# Surrogate key = klant_sk

logger.info("Start ETL Dim_Klant (SCD Type 2 met business key klantnr + source_id)")

# 1. Brondata ophalen
# We lezen klantdata uit beide bron-tabellen.
# Bij het inlezen voegen we direct een source_id toe,
# zodat elk record eenduidig herleidbaar is naar de bron.

query_accessoire_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Accessoire_Verkoop_Klant
"""

query_fiets_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Fiets_Verkoop_Klant
"""

df_accessoire_klant = pd.read_sql_query(query_accessoire_klant, sdm_conn)
df_fiets_klant = pd.read_sql_query(query_fiets_klant, sdm_conn)

# Voeg source_id toe
df_accessoire_klant["source_id"] = "Accessoire_Verkoop_Klant"
df_fiets_klant["source_id"] = "Fiets_Verkoop_Klant"

logger.info(f"Aantal klant-rijen uit accessoirebron: {len(df_accessoire_klant)}")
logger.info(f"Aantal klant-rijen uit fietsbron: {len(df_fiets_klant)}")


# 2. Brondata samenvoegen
# Omdat source_id nu onderdeel is van de business key,
# mogen dezelfde klantnummers uit verschillende bronnen
# gewoon naast elkaar bestaan.
# Daarom vervalt de oude conflictlogica volledig.
# We zetten de twee bronnen simpelweg onder elkaar.

df_klant_source = pd.concat(
    [df_accessoire_klant, df_fiets_klant],
    ignore_index=True
)

logger.info(f"Aantal records in samengestelde bronset: {len(df_klant_source)}")

# Controle op dubbelen binnen dezelfde business key
# (alleen handig als je wilt valideren dat dezelfde combinatie
# klantnr + source_id niet per ongeluk dubbel in de bron zit)
# dit komt echter niet voor in onze case, maar is misschien handig als er wel dubbele records binnenkomen
duplicate_bk = df_klant_source.duplicated(subset=["klantnr", "source_id"], keep=False)

if duplicate_bk.any():
    logger.warning("Er zijn dubbele records gevonden binnen dezelfde business key (klantnr + source_id).")
    logger.warning(f"Aantal dubbele bronrecords: {duplicate_bk.sum()}")


# 3. Actuele records uit het DWH ophalen
# Voor SCD Type 2 vergelijken we alleen met de actieve versie.
# Daarom halen we alleen records op met eindtijd IS NULL.

query_dwh_klant = """
SELECT
    klant_sk,
    klantnr,
    source_id,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum,
    begintijd,
    eindtijd
FROM Dim_Klant
WHERE eindtijd IS NULL
"""

df_klant_dwh = pd.read_sql_query(query_dwh_klant, dwh_conn)

logger.info(f"Aantal actuele klanten in DWH: {len(df_klant_dwh)}")


# 4. Helperfunctie: is de klant veranderd?
# We vergelijken alleen de inhoudelijke attributen.
# De business key zelf (klantnr + source_id) gebruiken we
# om het juiste actuele record te vinden.

def klant_is_changed(source_row, dwh_row):
    return (
        str(source_row["naam"]) != str(dwh_row["naam"]) or
        str(source_row["adres"]) != str(dwh_row["adres"]) or
        str(source_row["woonplaats"]) != str(dwh_row["woonplaats"]) or
        str(source_row["geslacht"]) != str(dwh_row["geslacht"]) or
        str(source_row["geboortedatum"]) != str(dwh_row["geboortedatum"])
    )

# 5. Vooraf bepalen hoeveel records nieuw / gewijzigd / ongewijzigd zijn
# Dit is handig voor logging en controle.
new_count = 0
changed_count = 0
preview_unchanged_count = 0

for _, src_row in df_klant_source.iterrows():
    klantnr = src_row["klantnr"]
    source_id = src_row["source_id"]

    # Zoek huidig actief record op basis van de nieuwe business key
    current_match = df_klant_dwh[
        (df_klant_dwh["klantnr"] == klantnr) &
        (df_klant_dwh["source_id"] == source_id)
    ]

    if current_match.empty:
        new_count += 1
    else:
        dwh_row = current_match.iloc[0]

        if klant_is_changed(src_row, dwh_row):
            changed_count += 1
        else:
            preview_unchanged_count += 1

logger.info(f"Aantal nieuwe klanten: {new_count}")
logger.info(f"Aantal gewijzigde klanten: {changed_count}")
logger.info(f"Aantal ongewijzigde klanten: {preview_unchanged_count}")


# 6. Echte ETL uitvoeren met SCD Type 2
# Regels:
# - nieuw business key record -> INSERT
# - bestaand business key record + wijziging -> UPDATE oude rij + INSERT nieuwe rij
# - bestaand business key record + geen wijziging -> niets doen

now_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

inserted_count = 0
updated_count = 0
unchanged_count = 0

logger.info("Start ETL voor Dim_Klant (SCD Type 2)")

for _, src_row in df_klant_source.iterrows():
    klantnr = src_row["klantnr"]
    source_id = src_row["source_id"]

    # Zoek match op business key: klantnr + source_id
    current_match = df_klant_dwh[ (df_klant_dwh["klantnr"] == klantnr) & (df_klant_dwh["source_id"] == source_id) ]

    # Als het een nieuwe rij is -> INSERT
    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Klant (
                klantnr,
                source_id,
                naam,
                adres,
                woonplaats,
                geslacht,
                geboortedatum,
                begintijd,
                eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (
            int(src_row["klantnr"]),
            src_row["source_id"],
            src_row["naam"],
            src_row["adres"],
            src_row["woonplaats"],
            src_row["geslacht"],
            src_row["geboortedatum"],
            now_ts
        ))

        inserted_count += 1


    # Bestaande rij -> check of gewijzigd
    else:
        dwh_row = current_match.iloc[0]

        if klant_is_changed(src_row, dwh_row):
            # Stap 1: oude actieve versie afsluiten
            dwh_conn.execute("""
                UPDATE Dim_Klant
                SET eindtijd = ?
                WHERE klant_sk = ?
            """, (
                now_ts,
                int(dwh_row["klant_sk"])
            ))

            # Stap 2: nieuwe versie toevoegen
            dwh_conn.execute("""
                INSERT INTO Dim_Klant (
                    klantnr,
                    source_id,
                    naam,
                    adres,
                    woonplaats,
                    geslacht,
                    geboortedatum,
                    begintijd,
                    eindtijd
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, NULL)
            """, (
                int(src_row["klantnr"]),
                src_row["source_id"],
                src_row["naam"],
                src_row["adres"],
                src_row["woonplaats"],
                src_row["geslacht"],
                src_row["geboortedatum"],
                now_ts
            ))

            updated_count += 1

        else:
            # Geen wijziging -> niets doen
            unchanged_count += 1


# 7. Commit
dwh_conn.commit()

logger.info(
    f"Dim_Klant klaar | inserted={inserted_count}, "
    f"updated_scd2={updated_count}, unchanged={unchanged_count}"
)

# 8. Controle
# Controleer de volledige inhoud van de dimensietabel.

result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Klant
    ORDER BY klantnr, source_id, klant_sk
""", dwh_conn)

print(result_df)

2026-04-01 12:05:31.578 | INFO     | __main__:<module>:15 - Start ETL Dim_Klant (SCD Type 2 met business key klantnr + source_id)
2026-04-01 12:05:31.584 | INFO     | __main__:<module>:51 - Aantal klant-rijen uit accessoirebron: 20
2026-04-01 12:05:31.585 | INFO     | __main__:<module>:52 - Aantal klant-rijen uit fietsbron: 25
2026-04-01 12:05:31.587 | INFO     | __main__:<module>:67 - Aantal records in samengestelde bronset: 45
2026-04-01 12:05:31.589 | INFO     | __main__:<module>:102 - Aantal actuele klanten in DWH: 0
2026-04-01 12:05:31.602 | INFO     | __main__:<module>:145 - Aantal nieuwe klanten: 45
2026-04-01 12:05:31.604 | INFO     | __main__:<module>:146 - Aantal gewijzigde klanten: 0
2026-04-01 12:05:31.605 | INFO     | __main__:<module>:147 - Aantal ongewijzigde klanten: 0
2026-04-01 12:05:31.606 | INFO     | __main__:<module>:162 - Start ETL voor Dim_Klant (SCD Type 2)
2026-04-01 12:05:31.626 | INFO     | __main__:<module>:250 - Dim_Klant klaar | inserted=45, updated_scd2=

    klant_sk  klantnr                 source_id               naam  \
0          1        1  Accessoire_Verkoop_Klant         Jan Jansen   
1         21        1       Fiets_Verkoop_Klant         Jan Jansen   
2          2        2  Accessoire_Verkoop_Klant     Sophie de Boer   
3         22        2       Fiets_Verkoop_Klant     Sophie de Boer   
4          3        3  Accessoire_Verkoop_Klant      Pieter Visser   
5         23        3       Fiets_Verkoop_Klant      Pieter Visser   
6          4        4  Accessoire_Verkoop_Klant          Emma Smit   
7         24        4       Fiets_Verkoop_Klant          Emma Smit   
8          5        5  Accessoire_Verkoop_Klant         Tom Bakker   
9         25        5       Fiets_Verkoop_Klant         Tom Bakker   
10         6        6  Accessoire_Verkoop_Klant        Lisa Meijer   
11        26        6       Fiets_Verkoop_Klant        Lisa Meijer   
12         7        7  Accessoire_Verkoop_Klant      Bart de Vries   
13        27        

Dim_Accessoire (SCD TYPE 1) ROBBERT

In [3]:
# Wat willen we bereiken?
# We willen Dim_Accessoire in het DWH vullen met SCD Type 1 logica:
# - nieuw accessoire in SDM, nog niet in DWH -> INSERT
# - bestaand accessoire, maar gewijzigd -> UPDATE (overschrijven)
# - bestaand accessoire, niet gewijzigd -> niets doen

# business key = accessoirenr uit SDM
# surrogate key = accessoire_sk in DWH

# 1. Brondata ophalen per bron (horizontale samenvoeging)
# Voor Dim_Accessoire komt de data uit:
# - Accessoire_Verkoop_Accessoire + Accessoire_Verkoop_Leverancier
# - Accessoire_Inkoop_Accessoire + Accessoire_Inkoop_Leverancier

# We JOINEN eerst per bron, omdat accessoire- en leveranciersdata
# in aparte tabellen staan.

query_accessoire_verkoop = """
SELECT
    a.accessoirenr,
    a.naam,
    a.soort,
    l.leveranciernr,
    l.naam AS leveranciernaam,
    l.adres AS leverancieradres,
    l.woonplaats AS leverancierplaats
FROM Accessoire_Verkoop_Accessoire a
JOIN Accessoire_Verkoop_Leverancier l
    ON a.leverancier = l.leveranciernr
"""

query_accessoire_inkoop = """
SELECT
    a.accessoirenr,
    a.naam,
    a.soort,
    l.leveranciernr,
    l.naam AS leveranciernaam,
    l.adres AS leverancieradres,
    l.woonplaats AS leverancierplaats
FROM Accessoire_Inkoop_Accessoire a
JOIN Accessoire_Inkoop_Leverancier l
    ON a.leverancier = l.leveranciernr
"""

df_accessoire_verkoop = pd.read_sql_query(query_accessoire_verkoop, sdm_conn)
df_accessoire_inkoop = pd.read_sql_query(query_accessoire_inkoop, sdm_conn)

logger.info(f"Aantal accessoire-rijen uit verkoopbron: {len(df_accessoire_verkoop)}")
logger.info(f"Aantal accessoire-rijen uit inkoopbron: {len(df_accessoire_inkoop)}")

# 2. Verticale samenvoeging tot één bronset
# We zetten de rijen uit verkoop en inkoop onder elkaar.
# Daarna houden we per business key nog maar één record over.

# Combineer beide bronnen
df_all = pd.concat(
    [
        df_accessoire_verkoop.assign(bron="verkoop"),
        df_accessoire_inkoop.assign(bron="inkoop")
    ],
    ignore_index=True
)

# Groepeer per accessoirenr
df_accessoire_source = []
conflicts = []

for accessoirenr, group in df_all.groupby("accessoirenr"):
    if len(group) == 1:
        # maar 1 bron → gewoon nemen
        df_accessoire_source.append(group.iloc[0])
    else:
        # meerdere bronnen → check of ze gelijk zijn
        unieke = group.drop_duplicates(subset=[
            "naam", "soort",
            "leveranciernr", "leveranciernaam",
            "leverancieradres", "leverancierplaats"
        ])

        if len(unieke) == 1:
            df_accessoire_source.append(unieke.iloc[0])
        else:
            conflicts.append(accessoirenr)
            logger.warning(f"CONFLICT accessoirenr={accessoirenr}")
            print(group)

# maak dataframe
df_accessoire_source = pd.DataFrame(df_accessoire_source)
logger.info(f"Aantal unieke accessoires in source: {len(df_accessoire_source)}")

# 3. Actuele records uit het DWH ophalen
# We vergelijken met de huidige inhoud van Dim_Accessoire.
query_dwh_accessoire = """
SELECT
    accessoire_sk,
    accessoirenr,
    naam,
    soort,
    leveranciernr,
    leveranciernaam,
    leverancieradres,
    leverancierplaats,
    begintijd,
    eindtijd
FROM Dim_Accessoire
WHERE eindtijd IS NULL
"""

df_accessoire_dwh = pd.read_sql_query(query_dwh_accessoire, dwh_conn)

logger.info(f"Aantal actuele accessoires in DWH: {len(df_accessoire_dwh)}")

# 4. Helperfunctie: is het accessoire veranderd?
# Bij SCD Type 1 vergelijken we eerst of de inhoudelijke attributen veranderd zijn.
# Alleen dan voeren we een UPDATE uit.

def accessoire_is_changed(source_row, dwh_row):
    return (
        str(source_row["naam"]) != str(dwh_row["naam"]) or
        str(source_row["soort"]) != str(dwh_row["soort"]) or
        str(source_row["leveranciernr"]) != str(dwh_row["leveranciernr"]) or
        str(source_row["leveranciernaam"]) != str(dwh_row["leveranciernaam"]) or
        str(source_row["leverancieradres"]) != str(dwh_row["leverancieradres"]) or
        str(source_row["leverancierplaats"]) != str(dwh_row["leverancierplaats"])
    )

# 5. Vooraf bepalen hoeveel records nieuw / gewijzigd / ongewijzigd zijn
new_count = 0
changed_count = 0
preview_unchanged_count = 0

for _, src_row in df_accessoire_source.iterrows():
    accessoirenr = src_row["accessoirenr"]

    current_match = df_accessoire_dwh[df_accessoire_dwh["accessoirenr"] == accessoirenr]

    if current_match.empty:
        new_count += 1
    else:
        dwh_row = current_match.iloc[0]

        if accessoire_is_changed(src_row, dwh_row):
            changed_count += 1
        else:
            preview_unchanged_count += 1

logger.info(f"Aantal nieuwe accessoires: {new_count}")
logger.info(f"Aantal gewijzigde accessoires: {changed_count}")
logger.info(f"Aantal ongewijzigde accessoires: {preview_unchanged_count}")

# 6. Echte ETL uitvoeren met SCD Type 1
# - nieuw -> INSERT
# - gewijzigd -> UPDATE
# - ongewijzigd -> niets doen

now_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

inserted_count = 0
updated_count = 0
unchanged_count = 0

logger.info("Start ETL voor Dim_Accessoire (SCD Type 1)")

for _, src_row in df_accessoire_source.iterrows():
    accessoirenr = src_row["accessoirenr"]

    current_match = df_accessoire_dwh[df_accessoire_dwh["accessoirenr"] == accessoirenr]

    # Nieuw accessoire -> INSERT
    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Accessoire (
                accessoirenr,
                naam,
                soort,
                leveranciernr,
                leveranciernaam,
                leverancieradres,
                leverancierplaats,
                begintijd,
                eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (
            int(src_row["accessoirenr"]),
            src_row["naam"],
            src_row["soort"],
            int(src_row["leveranciernr"]),
            src_row["leveranciernaam"],
            src_row["leverancieradres"],
            src_row["leverancierplaats"],
            now_ts
        ))
        inserted_count += 1
    # Bestaand accessoire -> check of gewijzigd
    else:
        dwh_row = current_match.iloc[0]

        if accessoire_is_changed(src_row, dwh_row):
            # Bij SCD Type 1 overschrijven we de bestaande rij.
            dwh_conn.execute("""
                UPDATE Dim_Accessoire
                SET
                    naam = ?,
                    soort = ?,
                    leveranciernr = ?,
                    leveranciernaam = ?,
                    leverancieradres = ?,
                    leverancierplaats = ?
                WHERE accessoire_sk = ?
            """, (
                src_row["naam"],
                src_row["soort"],
                int(src_row["leveranciernr"]),
                src_row["leveranciernaam"],
                src_row["leverancieradres"],
                src_row["leverancierplaats"],
                int(dwh_row["accessoire_sk"])
            ))
            updated_count += 1
        else:
            unchanged_count += 1

dwh_conn.commit()

logger.info(
    f"Dim_Accessoire klaar | inserted={inserted_count}, "
    f"updated_scd1={updated_count}, unchanged={unchanged_count}"
)

# 7. Controle
result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Accessoire
    ORDER BY accessoirenr, accessoire_sk
""", dwh_conn)

print(result_df)

2026-04-01 12:06:30.913 | INFO     | __main__:<module>:49 - Aantal accessoire-rijen uit verkoopbron: 10
2026-04-01 12:06:30.914 | INFO     | __main__:<module>:50 - Aantal accessoire-rijen uit inkoopbron: 13
2026-04-01 12:06:30.944 | INFO     | __main__:<module>:90 - Aantal unieke accessoires in source: 13
2026-04-01 12:06:30.946 | INFO     | __main__:<module>:112 - Aantal actuele accessoires in DWH: 0
2026-04-01 12:06:30.951 | INFO     | __main__:<module>:148 - Aantal nieuwe accessoires: 13
2026-04-01 12:06:30.952 | INFO     | __main__:<module>:149 - Aantal gewijzigde accessoires: 0
2026-04-01 12:06:30.955 | INFO     | __main__:<module>:150 - Aantal ongewijzigde accessoires: 0
2026-04-01 12:06:30.958 | INFO     | __main__:<module>:163 - Start ETL voor Dim_Accessoire (SCD Type 1)
2026-04-01 12:06:30.974 | INFO     | __main__:<module>:227 - Dim_Accessoire klaar | inserted=13, updated_scd1=0, unchanged=0


    accessoire_sk  accessoirenr                      naam        soort  \
0              14             1              LED voorlamp  Verlichting   
1              15             2           LED achterlicht  Verlichting   
2              16             3  USB-oplaadbare fietslamp  Verlichting   
3              17             4              Ringslot AXA       Sloten   
4              18             5    Ketting met cijferslot       Sloten   
5              19             6         U-slot met beugel       Sloten   
6              20             7    Dubbele fietstas zwart       Tassen   
7              21             8   Enkele schouderfietstas       Tassen   
8              22             9      Waterdichte zadeltas       Tassen   
9              23            10      Kinderfietsmand roze       Tassen   
10             24            11  Reflecterende spaakclips  Verlichting   
11             25            12        Vouwbaar kabelslot       Sloten   
12             26            13       

Dim_Datum ROBBERT

In [4]:
# Wat willen we bereiken?
# We willen Dim_Datum in het DWH vullen.
# Voor Dim_Datum geldt:
# - nieuwe datum in SDM, nog niet in DWH -> INSERT
# - bestaande datum in DWH -> niets doen
# Voor Dim_Datum gebruiken we geen SCD Type 1 of Type 2,
# omdat datumwaarden zelf niet wijzigen.

# business key = datum
# surrogate key = datum_sk in DWH

# 1. Brondata ophalen uit de relevante feitbronnen=
# We halen de datums op uit:
# - Fiets_Verkoop
# - Accessoire_Verkoop
# - Onderhoud
# Voor Fct_Inkoop hebben we in het SDM geen echte datumkolom,
# maar maand + jaar. Daarom gebruiken we hier alleen de tabellen
# die een volledige datum bevatten.

query_fiets_verkoop_datum = """
SELECT datum
FROM Fiets_Verkoop
"""

query_accessoire_verkoop_datum = """
SELECT datum
FROM Accessoire_Verkoop
"""

query_onderhoud_datum = """
SELECT datum
FROM Onderhoud
"""

df_fiets_verkoop_datum = pd.read_sql_query(query_fiets_verkoop_datum, sdm_conn)
df_accessoire_verkoop_datum = pd.read_sql_query(query_accessoire_verkoop_datum, sdm_conn)
df_onderhoud_datum = pd.read_sql_query(query_onderhoud_datum, sdm_conn)

logger.info(f"Aantal datum-rijen uit Fiets_Verkoop: {len(df_fiets_verkoop_datum)}")
logger.info(f"Aantal datum-rijen uit Accessoire_Verkoop: {len(df_accessoire_verkoop_datum)}")
logger.info(f"Aantal datum-rijen uit Onderhoud: {len(df_onderhoud_datum)}")

# 2. Verticale samenvoeging tot één bronset
# We zetten alle datumrijen onder elkaar.
# Daarna houden we alleen unieke datums over.

df_datum_source = pd.concat(
    [df_fiets_verkoop_datum, df_accessoire_verkoop_datum, df_onderhoud_datum],
    ignore_index=True
)

df_datum_source = df_datum_source.drop_duplicates(subset=["datum"]).reset_index(drop=True)

logger.info(f"Aantal unieke datums in source: {len(df_datum_source)}")

# 3. Datumonderdelen afleiden
# We zetten de datumkolom om naar datetime,
# zodat we year, month en day kunnen afleiden.

df_datum_source["datum"] = pd.to_datetime(df_datum_source["datum"])
df_datum_source["year"] = df_datum_source["datum"].dt.year
df_datum_source["month"] = df_datum_source["datum"].dt.month
df_datum_source["day"] = df_datum_source["datum"].dt.day

# Voor opslag in SQLite zetten we datum weer om naar YYYY-MM-DD string
df_datum_source["datum"] = df_datum_source["datum"].dt.strftime("%Y-%m-%d")

# 4. Bestaande datums uit DWH ophalen
query_dwh_datum = """
SELECT
    datum_sk,
    datum,
    year,
    month,
    day
FROM Dim_Datum
"""

df_datum_dwh = pd.read_sql_query(query_dwh_datum, dwh_conn)

logger.info(f"Aantal datums in DWH: {len(df_datum_dwh)}")

# 5. Bepalen welke datums nieuw zijn
new_count = 0
existing_count = 0

for _, src_row in df_datum_source.iterrows():
    datum = src_row["datum"]

    current_match = df_datum_dwh[df_datum_dwh["datum"] == datum]

    if current_match.empty:
        new_count += 1
    else:
        existing_count += 1

logger.info(f"Aantal nieuwe datums: {new_count}")
logger.info(f"Aantal bestaande datums: {existing_count}")

# 6. Echte ETL uitvoeren
# Alleen nieuwe datums worden toegevoegd.

inserted_count = 0
logger.info("Start ETL voor Dim_Datum")

for _, src_row in df_datum_source.iterrows():
    datum = src_row["datum"]

    current_match = df_datum_dwh[df_datum_dwh["datum"] == datum]

    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Datum (
                datum,
                year,
                month,
                day
            )
            VALUES (?, ?, ?, ?)
        """, (
            src_row["datum"],
            int(src_row["year"]),
            int(src_row["month"]),
            int(src_row["day"])
        ))
        inserted_count += 1

dwh_conn.commit()
logger.info(f"Dim_Datum klaar | inserted={inserted_count}")

# 7. Controle

result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Datum
    ORDER BY datum
""", dwh_conn)

print(result_df)

2026-04-01 12:06:38.712 | INFO     | __main__:<module>:40 - Aantal datum-rijen uit Fiets_Verkoop: 150
2026-04-01 12:06:38.714 | INFO     | __main__:<module>:41 - Aantal datum-rijen uit Accessoire_Verkoop: 100
2026-04-01 12:06:38.715 | INFO     | __main__:<module>:42 - Aantal datum-rijen uit Onderhoud: 50
2026-04-01 12:06:38.717 | INFO     | __main__:<module>:55 - Aantal unieke datums in source: 196
2026-04-01 12:06:38.730 | INFO     | __main__:<module>:82 - Aantal datums in DWH: 0
2026-04-01 12:06:38.760 | INFO     | __main__:<module>:98 - Aantal nieuwe datums: 196
2026-04-01 12:06:38.761 | INFO     | __main__:<module>:99 - Aantal bestaande datums: 0
2026-04-01 12:06:38.762 | INFO     | __main__:<module>:105 - Start ETL voor Dim_Datum
2026-04-01 12:06:38.795 | INFO     | __main__:<module>:130 - Dim_Datum klaar | inserted=196


     datum_sk       datum  year  month  day
0         207  2024-01-02  2024      1    2
1         324  2024-01-06  2024      1    6
2         221  2024-01-08  2024      1    8
3         369  2024-01-10  2024      1   10
4         286  2024-01-11  2024      1   11
..        ...         ...   ...    ...  ...
191       262  2024-12-26  2024     12   26
192       384  2024-12-27  2024     12   27
193       373  2024-12-28  2024     12   28
194       296  2024-12-30  2024     12   30
195       320  2024-12-31  2024     12   31

[196 rows x 5 columns]


Fct_Verkoop ROBBERT


In [5]:
# Wat willen we bereiken?
# We willen Fct_Verkoop vullen met verkoopgebeurtenissen uit:
# - Fiets_Verkoop
# - Accessoire_Verkoop

# In het DWH komt dit samen in één feittabel:
# Fct_Verkoop
# De feittabel bevat:
# - verkoopnr
# - verkoop_type
# - datum_sk
# - aantal
# - totaalprijs
# - standaardprijs
# - klant_sk
# - fiets_sk of accessoire_sk
# - monteur_sk

# 1. Brondata ophalen uit beide verkoopbronnen
# We voegen nu ook verkoop_type toe, zodat de samengestelde PK
# (verkoopnr, verkoop_type) correct gevuld kan worden.
logger.info("Ophalen Verkoop uit SDM")

# --- BRONDATA ---
query_fiets_verkoop = """
SELECT
    fv.datum,
    fv.aantal,
    fv.verkoopprijs AS totaalprijs,
    ff.standaardprijs,
    fv.klant,
    fv.fiets,
    fv.monteur,
    NULL AS accessoire
FROM Fiets_Verkoop fv
JOIN Fiets_Verkoop_Fiets ff
    ON fv.fiets = ff.fietsnr
"""

query_accessoire_verkoop = """
SELECT
    av.datum,
    av.aantal,
    av.verkoopprijs AS totaalprijs,
    aa.standaardprijs,
    av.klant,
    NULL AS fiets,
    av.monteur,
    av.accessoire
FROM Accessoire_Verkoop av
JOIN Accessoire_Verkoop_Accessoire aa
    ON av.accessoire = aa.accessoirenr
"""

df_verkoop_source = pd.concat([
    pd.read_sql_query(query_fiets_verkoop, sdm_conn),
    pd.read_sql_query(query_accessoire_verkoop, sdm_conn)
], ignore_index=True)

logger.info(f"Totaal bron: {len(df_verkoop_source)}")

# --- DIMENSIES ---
df_datum_dwh = pd.read_sql_query("SELECT datum_sk, datum FROM Dim_Datum", dwh_conn)

df_klant_dwh = pd.read_sql_query("""
SELECT klant_sk, klantnr FROM Dim_Klant WHERE eindtijd IS NULL
""", dwh_conn)

df_fiets_dwh = pd.read_sql_query("""
SELECT fiets_sk, fietsnr FROM Dim_Fiets WHERE eindtijd IS NULL
""", dwh_conn)

df_accessoire_dwh = pd.read_sql_query("""
SELECT accessoire_sk, accessoirenr FROM Dim_Accessoire WHERE eindtijd IS NULL
""", dwh_conn)

df_monteur_dwh = pd.read_sql_query("""
SELECT monteur_sk, monteurnr FROM Dim_Monteur WHERE eindtijd IS NULL
""", dwh_conn)

# --- BESTAANDE ---
df_verkoop_dwh = pd.read_sql_query("""
SELECT verkoop_type, datum_sk, aantal, klant_sk, fiets_sk, accessoire_sk
FROM Fct_Verkoop
""", dwh_conn)

bestaande_records = set(
    tuple(x) for x in df_verkoop_dwh.to_records(index=False)
)

# --- SAFE HELPERS ---
def safe_lookup(df, key_col, key_val, return_col, naam):
    match = df[df[key_col] == key_val]
    if match.empty:
        raise ValueError(f"{naam} niet gevonden: {key_val}")
    return int(match.iloc[0][return_col])

def get_datum_sk(d):
    return safe_lookup(df_datum_dwh, "datum", str(d), "datum_sk", "datum")

def get_klant_sk(k):
    return safe_lookup(df_klant_dwh, "klantnr", int(k), "klant_sk", "klant")

def get_fiets_sk(f):
    match = df_fiets_dwh[df_fiets_dwh["fietsnr"] == int(f)]
    return int(match.iloc[0]["fiets_sk"]) if not match.empty else None

def get_accessoire_sk(a):
    match = df_accessoire_dwh[df_accessoire_dwh["accessoirenr"] == int(a)]
    return int(match.iloc[0]["accessoire_sk"]) if not match.empty else None

def get_monteur_sk(m):
    return safe_lookup(df_monteur_dwh, "monteurnr", int(m), "monteur_sk", "monteur")

# --- ETL ---
logger.info("Start ETL Fct_Verkoop")

inserted = 0
skipped = 0
errors = 0

for _, row in df_verkoop_source.iterrows():
    try:
        datum_sk = get_datum_sk(row["datum"])
        klant_sk = get_klant_sk(row["klant"])
        monteur_sk = get_monteur_sk(row["monteur"])

        # type bepalen
        if pd.notna(row["fiets"]):
            verkoop_type = "fiets"
            fiets_sk = get_fiets_sk(row["fiets"])
            accessoire_sk = None
        else:
            verkoop_type = "accessoire"
            fiets_sk = None
            accessoire_sk = get_accessoire_sk(row["accessoire"])

        # validatie
        if (fiets_sk is None and accessoire_sk is None) or \
           (fiets_sk is not None and accessoire_sk is not None):
            logger.warning(f"Conflict: {row}")
            continue

        record_key = (
            verkoop_type,
            datum_sk,
            int(row["aantal"]),
            klant_sk,
            fiets_sk,
            accessoire_sk
        )

        if record_key in bestaande_records:
            skipped += 1
            continue

        dwh_conn.execute("""
            INSERT INTO Fct_Verkoop (
                verkoop_type,
                datum_sk,
                aantal,
                totaalprijs,
                standaardprijs,
                klant_sk,
                fiets_sk,
                accessoire_sk,
                monteur_sk
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            verkoop_type,
            datum_sk,
            int(row["aantal"]),
            float(row["totaalprijs"]),
            float(row["standaardprijs"]),
            klant_sk,
            fiets_sk,
            accessoire_sk,
            monteur_sk
        ))

        bestaande_records.add(record_key)  
        inserted += 1

    except Exception as e:
        logger.error(f"Fout bij row {row}: {e}")
        errors += 1

dwh_conn.commit()

logger.info(f"Fct_Verkoop klaar inserted={inserted}, skipped={skipped}, errors={errors}")

2026-04-01 12:06:44.394 | INFO     | __main__:<module>:22 - Ophalen Verkoop uit SDM
2026-04-01 12:06:44.400 | INFO     | __main__:<module>:60 - Totaal bron: 250
2026-04-01 12:06:44.407 | INFO     | __main__:<module>:116 - Start ETL Fct_Verkoop
2026-04-01 12:06:44.411 | ERROR    | __main__:<module>:186 - Fout bij row datum             2024-03-09
aantal                     3
totaalprijs          1020.48
standaardprijs        887.87
klant                     25
fiets                     61
monteur                    6
accessoire              None
Name: 0, dtype: object: monteur niet gevonden: 6
2026-04-01 12:06:44.413 | ERROR    | __main__:<module>:186 - Fout bij row datum             2024-07-12
aantal                     1
totaalprijs          1391.84
standaardprijs        551.27
klant                     23
fiets                     58
monteur                    5
accessoire              None
Name: 1, dtype: object: monteur niet gevonden: 5
2026-04-01 12:06:44.416 | ERROR    | __main__: